In [ ]:
from pyspark.sql.functions import col

from olist_silver.transformations import (
    is_valid_uuid,
    merge_into,
    with_processed_timestamp,
)

In [ ]:
catalog = dbutils.widgets.get("catalog")

bronze_schema = dbutils.widgets.get("bronze_schema")
raw_olist_order_payments_table_name = dbutils.widgets.get("raw_olist_order_payments_table")

silver_schema = dbutils.widgets.get("silver_schema")
order_payments_table_name = dbutils.widgets.get("order_payments_table")

In [ ]:
raw_olist_order_payments_df = spark.table(f"{catalog}.{bronze_schema}.{raw_olist_order_payments_table_name}")

In [ ]:
if not spark.catalog.tableExists(f"{catalog}.{silver_schema}.{order_payments_table_name}"):
    spark.sql(
        f"""
        CREATE TABLE {catalog}.{silver_schema}.{order_payments_table_name} (
            orderId STRING,
            paymentSequential INT,
            paymentType STRING,
            paymentInstallments INT,
            paymentValue DOUBLE,
            processedTimestamp TIMESTAMP
        )
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
        """
    )

In [ ]:
order_payments_silver_df = with_processed_timestamp(
    raw_olist_order_payments_df.where(is_valid_uuid("order_id") & col("payment_sequential").cast("int").isNotNull())
    .select(
        col("order_id").cast("string").alias("orderId"),
        col("payment_sequential").cast("int").alias("paymentSequential"),
        col("payment_type").cast("string").alias("paymentType"),
        col("payment_installments").cast("int").alias("paymentInstallments"),
        col("payment_value").cast("double").alias("paymentValue"),
    )
    .dropDuplicates(["orderId", "paymentSequential"])
)

In [ ]:
merge_into(
    spark,
    target=f"{catalog}.{silver_schema}.{order_payments_table_name}",
    source_view="order_payments_silver_view",
    keys=["orderId", "paymentSequential"],
    source_df=order_payments_silver_df,
)